# 🚀 DiffMM + Flow Matching — Google Colab

Notebook này chạy thí nghiệm so sánh **3 mô hình khuyến nghị đa phương thức**:
- **DiffMM**: Diffusion model Markov gốc
- **Flow Matching Original**: Thay SDE bằng ODE
- **Flow Matching Optimized**: Vector field liên tục + Euler solver

trên **3 dataset**: Baby, Sports, TikTok.

---
⚠️ **Trước khi chạy**: Đảm bảo Runtime → Change runtime type → **GPU** (T4 hoặc A100)


## 📦 Cell 1: Kiểm tra GPU & Mount Google Drive

In [ ]:
import torch

# Kiểm tra GPU
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Không có GPU! Vào Runtime → Change runtime type → GPU')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive đã mount tại /content/drive')

## 📥 Cell 2: Clone Code từ GitHub & Cài Dependencies

In [ ]:
import os

# ══════════════════════════════════════════════════════════
# 🔧 THAY ĐỔI: URL GitHub repo của bạn
    GITHUB_REPO = 'https://github.com/thyelmot/VI_DiffMM_Colab.git'
# ══════════════════════════════════════════════════════════

    WORK_DIR = '/content/VI_DiffMM_Colab'

if os.path.exists(WORK_DIR):
    print('Folder đã tồn tại, đang pull bản mới nhất...')
    os.system(f'cd {WORK_DIR} && git pull')
else:
    print('Clone repository...')
    os.system(f'git clone {GITHUB_REPO} {WORK_DIR}')

os.chdir(WORK_DIR)
print(f'✅ Working directory: {os.getcwd()}')
print('Files:', os.listdir('.'))

In [ ]:
# Cài dependencies
!pip install -q -r requirements.txt
print('✅ Dependencies đã cài xong')

## 🗂️ Cell 3: Cấu hình Đường Dẫn Dataset & Checkpoint

> **Hướng dẫn**: Upload thư mục `Datasets/` lên Google Drive theo cấu trúc:
> ```
> MyDrive/
> └── DiffMM_Data/
>     ├── baby/
>     │   ├── trnMat.pkl
>     │   ├── tstMat.pkl
>     │   ├── image_feat.npy
>     │   └── text_feat.npy
>     ├── sports/   (tương tự)
>     └── tiktok/   (+ audio_feat.npy)
> ```

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 🔧 THAY ĐỔI CÁC ĐƯỜNG DẪN NÀY CHO PHÙ HỢP VỚI DRIVE CỦA BẠN
# ══════════════════════════════════════════════════════════════════

# Đường dẫn tới thư mục chứa baby/, sports/, tiktok/ trên Drive
DATASET_PATH = '/content/drive/MyDrive/DiffMM_Data'

# Thư mục lưu checkpoint (.pth) và kết quả (.json) — sẽ tự tạo
CHECKPOINT_DIR = '/content/drive/MyDrive/DiffMM_Checkpoints'

# File log — None để không lưu, hoặc đặt đường dẫn
LOG_FILE = f'{CHECKPOINT_DIR}/training.log'  # hoặc None

# ══════════════════════════════════════════════════════════════════

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Kiểm tra dataset tồn tại
for ds in ['baby', 'sports', 'tiktok']:
    ds_path = os.path.join(DATASET_PATH, ds)
    if os.path.exists(ds_path):
        files = os.listdir(ds_path)
        print(f'✅ {ds}: {files}')
    else:
        print(f'❌ {ds}: KHÔNG TÌM THẤY tại {ds_path}')

## ⚙️ Cell 4: Cấu Hình Hyperparameter

Chỉnh sửa `config` bên dưới theo nhu cầu:

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 🔧 CHỈNH SỬA HYPERPARAMETER TẠI ĐÂY
# ══════════════════════════════════════════════════════════════════

config = {
    # Dataset: 'baby' | 'sports' | 'tiktok'
    'data': 'baby',

    # Mô hình: 'diffmm' | 'flowmatching_original' | 'flowmatching_optimized'
    'model_type': 'flowmatching_optimized',

    # Số epoch (50 để reproduce, 2 để test nhanh)
    'epoch': 50,

    # Hyperparameters (giá trị tốt nhất theo paper)
    'lr': 1e-3,
    'batch': 1024,
    'tstBat': 256,
    'latdim': 64,
    'gnn_layer': 1,
    'topk': 20,
    'keepRate': 1.0,    # baby: 1.0 | sports: 0.5 | tiktok: 0.5
    'ssl_reg': 1e-1,    # baby: 1e-1 | sports: 1e-2 | tiktok: 1e-2
    'temp': 0.5,
    'reg': 1e-5,        # baby/sports: 1e-5 | tiktok: 1e-4
    'e_loss': 0.01,     # baby/sports: 0.01 | tiktok: 0.1
    'cl_method': 0,     # baby/sports: 0 | tiktok: 1
    'trans': 0,         # baby/sports: 0 | tiktok: 1
    'rebuild_k': 1,
    'ris_lambda': 0.5,
    'ris_adj_lambda': 0.2,
    'dims': '[1000]',
    'd_emb_size': 10,
    'norm': False,
    'steps': 5,
    'noise_scale': 0.1,
    'noise_min': 0.0001,
    'noise_max': 0.02,
    'sampling_noise': False,
    'sampling_steps': 0,
    'seed': 421,
    'gpu': '0',
    'tstEpoch': 1,

    # Paths (tự động lấy từ các ô trên)
    'dataset_path': DATASET_PATH,
    'checkpoint_dir': CHECKPOINT_DIR,
    'log_file': LOG_FILE,
}

print('✅ Config:')
for k, v in config.items():
    print(f'   {k:20s} = {v}')

## 🏋️ Cell 5: Bắt Đầu Training

> 💡 **Auto-Resume**: Nếu Colab bị ngắt, chỉ cần chạy lại cell này — training sẽ tiếp tục từ checkpoint cuối cùng được lưu trên Drive.

In [ ]:
import sys
import os

# Đảm bảo working dir đúng
os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

from Main import train_colab

print(f'🚀 Bắt đầu training: [{config["model_type"]}] trên [{config["data"]}]')
print(f'   Checkpoint sẽ lưu tại: {CHECKPOINT_DIR}')
print('='*60)

results = train_colab(
    config_dict=config,
    dataset_base_path=DATASET_PATH
)

print('\n' + '='*60)
print('🎉 Training hoàn thành!')
print(f"   Best Epoch : {results['best_epoch']}")
print(f"   Recall@20  : {results['recall']:.6f}")
print(f"   NDCG@20    : {results['ndcg']:.6f}")
print(f"   Precision  : {results['precision']:.6f}")

## 📊 Cell 6: Benchmark Toàn Diện (3 Model × 3 Dataset)

> ⏱️ **Lưu ý**: Benchmark đầy đủ (50 epoch × 9 experiments) mất rất lâu.
> Dùng `epochs=2` để kiểm tra nhanh, sau đó chạy full với `epochs=50`.

In [ ]:
from Benchmark import run_benchmark_from_notebook
import pandas as pd

# ══════════════════════════════════════════════════════════════════
# 🔧 Tùy chỉnh benchmark
BENCHMARK_EPOCHS = 2            # 2 để test nhanh, 50 để full
BENCHMARK_DATASETS = ['tiktok', 'baby', 'sports']  # Có thể chọn subset
BENCHMARK_MODELS = ['diffmm', 'flowmatching_original', 'flowmatching_optimized']
# ══════════════════════════════════════════════════════════════════

print(f'🏁 Bắt đầu Benchmark: {BENCHMARK_MODELS}')
print(f'   Datasets: {BENCHMARK_DATASETS}, Epochs: {BENCHMARK_EPOCHS}')
print('='*60)

df_results = run_benchmark_from_notebook(
    datasets=BENCHMARK_DATASETS,
    models=BENCHMARK_MODELS,
    epochs=BENCHMARK_EPOCHS,
    gpu='0',
    dataset_path=DATASET_PATH,
    checkpoint_dir=CHECKPOINT_DIR
)

print('\n\n' + '='*80)
print('FINAL BENCHMARK RESULTS'.center(80))
print('='*80)
if not df_results.empty:
    display(df_results)
    
    # Lưu CSV
    csv_path = os.path.join(CHECKPOINT_DIR, f'benchmark_results_ep{BENCHMARK_EPOCHS}.csv')
    df_results.to_csv(csv_path, index=False)
    print(f'\n✅ Kết quả đã lưu tại: {csv_path}')
else:
    print('❌ Không có kết quả')

## 🔍 Cell 7: Xem kết quả từ Checkpoint có sẵn

In [ ]:
import json
import os
import glob

# Đọc tất cả file results_*.json từ checkpoint dir
result_files = glob.glob(os.path.join(CHECKPOINT_DIR, 'results_*.json'))

if result_files:
    rows = []
    for f in sorted(result_files):
        basename = os.path.basename(f).replace('results_', '').replace('.json', '')
        parts = basename.rsplit('_', 1)
        model = '_'.join(parts[:-1]) if len(parts) > 1 else basename
        dataset = parts[-1] if len(parts) > 1 else '?'
        with open(f) as fp:
            res = json.load(fp)
        rows.append({
            'Model': model,
            'Dataset': dataset.upper(),
            'Recall@20': round(res.get('recall', 0), 6),
            'NDCG@20': round(res.get('ndcg', 0), 6),
            'Precision@20': round(res.get('precision', 0), 6),
            'Best Epoch': res.get('best_epoch', '?'),
        })
    
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df)
else:
    print(f'Chưa có file kết quả trong {CHECKPOINT_DIR}')
    print('Hãy chạy training trước (Cell 5 hoặc Cell 6)')

---
## 📋 Tóm Tắt Hyperparameter Tốt Nhất Theo Dataset

| Tham số | Baby | Sports | TikTok |
|---|---|---|---|
| `reg` | `1e-5` | `1e-5` | `1e-4` |
| `ssl_reg` | `1e-1` | `1e-2` | `1e-2` |
| `keepRate` | `1.0` | `0.5` | `0.5` |
| `e_loss` | `0.01` | `0.01` | `0.1` |
| `trans` | `0` | `0` | `1` |
| `cl_method` | `0` | `0` | `1` |

## 🔄 Quy Trình Khi Colab Bị Ngắt

1. Kết nối lại Colab runtime
2. Chạy lại **Cell 1** (mount Drive)
3. Chạy lại **Cell 2** (cd vào thư mục code)
4. Chạy lại **Cell 3** (set paths)
5. Chạy thẳng **Cell 5** — training sẽ **tự động resume** từ checkpoint!
